# SHIPP Stage 9C — Retrieval evaluation (spec §11.3)

Runs every case in `rag/eval/eval_set.json` through `search_listing_context()` and appends one row per
case to `shipp_gold.rag_eval_results`. Metrics: hit@K and MRR over positive cases, latency; negative cases
report their top score. **The eval set must be updated whenever the demo seed changes** — the notebook stops
if it references a listing that is not in Gold.

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `rag_eval_results`.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

In [ ]:
import uuid

from rag.evaluation import hit_at_k, load_eval_set, reciprocal_rank, summarize
from rag.retrieval import search_listing_context

RESULTS_SCHEMA = ("run_id string, evaluated_at timestamp, case_id string, query string, expected string, "
                  "retrieved string, top_score double, hit int, rr double, latency_ms long, positive boolean, "
                  "k int, index_name string")
cases = load_eval_set(os.path.join(REPO_ROOT, "rag", "eval", "eval_set.json"))
gold_ids = {r.listing_id for r in spark.table(GOLD_LISTING_SEARCH_DOCS).select("listing_id").collect()}
stale = [(c["id"], x) for c in cases for x in c["expected_listing_ids"] if x not in gold_ids]
assert not stale, f"FAILED: eval set references listings not in Gold: {stale} — update rag/eval/eval_set.json"
print(f"{len(cases)} cases, K = {RAG_EVAL_K}")

In [ ]:
run_id, rows, results = str(uuid.uuid4()), [], []
for c in cases:
    t0 = time.time()
    hits = search_listing_context(c["query"], k=RAG_EVAL_K, workspace_client=w)
    latency = int((time.time() - t0) * 1000)
    retrieved = [h["listing_id"] for h in hits]
    positive = bool(c["expected_listing_ids"])
    hit = hit_at_k(retrieved, c["expected_listing_ids"], RAG_EVAL_K) if positive else 0
    rr = reciprocal_rank(retrieved, c["expected_listing_ids"]) if positive else 0.0
    top = hits[0]["score"] if hits else None
    results.append({"hit": hit, "rr": rr, "latency_ms": latency, "positive": positive})
    rows.append((run_id, None, c["id"], c["query"], ",".join(c["expected_listing_ids"]), ",".join(retrieved),
                 top, hit, rr, latency, positive, RAG_EVAL_K, VS_INDEX_NAME))
    print(f"{c['id']}  hit={hit}  rr={rr:.2f}  top_score={top}  {latency} ms  → {retrieved}")

(spark.createDataFrame(rows, RESULTS_SCHEMA).withColumn("evaluated_at", F.current_timestamp())
 .write.format("delta").mode("append").saveAsTable(RAG_EVAL_RESULTS))

In [ ]:
summary = summarize(results)
print(summary)
assert summary["positive_cases"] > 0, "FAILED: no positive eval cases"
assert summary["hit_rate"] >= RAG_EVAL_MIN_HIT_RATE, (
    f"FAILED: hit@{RAG_EVAL_K} = {summary['hit_rate']} < {RAG_EVAL_MIN_HIT_RATE}. "
    "Inspect search_text in Gold before touching the threshold."
)
print("PASS — retrieval quality meets the threshold.")
dbutils.notebook.exit(json.dumps({"table": RAG_EVAL_RESULTS, "run_id": run_id, **summary, "status": "PASS"}))